# Week 4: Generating Sonnets using Lists, Functions, Conditionals and Iteration

Last week we began working toward the goal of generating or recognizing a poem with a defined meter and rhyme scheme. We used the [CMU Pronouncing Dictionary](http://www.speech.cs.cmu.edu/cgi-bin/cmudict) which allows us to look up the pronuciation of a word and extract its stress pattern.

We introduced a lot of new programming concepts last week, so the plan is to mostly review them this week, but we do have a few new ones.

New Python concepts:
- List slicing
- Defining functions
- The `in` operator 
- The `not` operator
- The string method `.join()`
- The list method `.append()`

### Links

You may find these sections of Melanie Walsh's textbook useful:
* [Python Comparisons and Conditionals](https://melaniewalsh.github.io/Intro-Cultural-Analytics/02-Python/08-Comparisons-Conditionals.html)
* [Python Links and Loops](https://melaniewalsh.github.io/Intro-Cultural-Analytics/02-Python/09-Lists-Loops-Part1.html)



Today our goal is to generate poetry in the form of a sonnet. We will focus on the Shakespearean form which has 14 lines with a meter of iambic pentameter. Eventually, we will also add in a rhyme scheme, but our first goal will be generating lines in iambic pentameter.

An _iamb_ is a metrical foot where the first syllable is unstressed and the second is stressed. Since we can extract the stress pattern for words in our dictionary, we can randomly select words until we find a matching stress pattern. We will start simple and try a few different approaches to generate a sonnet that has the correct stress pattern and even a valid rhyme scheme. How optimisitic are you that the generated sonnets will make any sense at all?

Before we start to work on different approaches, we need to do some setup:
1. We need a corpus or list of words from which we will select to generate a sonnet.
2. We need to be able to look up the pronunciation of a word to discover its stress pattern. (We did most of this last week.)
3. We need to be able to compare the stress patterns of different words.



## 1. Creating a corpus of words

For all of our approaches we will need a list of words from which to select. We could use the full dictionary, but a smaller corpus might give our results a specific flavour, and will also reduce the computation needed. If we draw our words from a list of words that Shakespeare used to write his sonnets, maybe our generated poems will sound Shakespearean.

For this lecture, we will use this version of [Shakespeare's sonnets](https://www.gutenberg.org/cache/epub/1041/pg1041.txt) as our text. [Project Gutenberg](https://www.gutenberg.org/) is a collection of texts that are no longer under copyright, making this text easy to work with.

Next week we will talk about how to open and read files.  We will also spend more time talking about the data science problem of _cleaning data_. For now however, you should run the following code cell without expecting to fully understand it.

The code below reads the file downloaded from the Gutenberg Project and creates a list of poem lines by excluding blank lines, poem numbers, and licence boiler plate. This file is in the same folder as this notebook, so you can take a look at it to see what it contains.

In [ ]:
import re

# open the file "Shakespeare_Sonnets.txt" for reading and store its contents as 
# a list of lines in the variable lines
with open("Shakespeare_Sonnets.txt", encoding="utf-8") as f:
    lines = f.readlines()

# After looking at the contents of the file, we notice that lines 0-34
# contain license boilerplate and do not contain any poetry.  Also, the
# lines from 2654 to the end of the file do not contain any poetry.
# So, we will take a slice of the data from line 34 up to but not including
# line 2654, since these are the lines that contain poetry or the numbering
# for the sonnets (in roman numerals).

raw_lines = lines[34:2654]

# A list to accumulate usable lines from the poems
poem_text_lines = []


for line in raw_lines:

    # remove the whitespace from the beginning and end of the line
    stripped_line = line.strip()

    # If stripped is empty, then it was a blank line, so we will skip it
    if not stripped_line:
        continue
    # If stripped is all upper case then skip it.
    elif stripped_line == stripped_line.upper():
        continue
    else:
        # remove punctuation (except apostrophes) and set to lower case
        cleaned_line = re.sub(r"[^a-z'\u2019]", " ", line.lower())
        poem_text_lines.append(cleaned_line)



print(len(poem_text_lines), "usable lines of verse")

# Check a line to see what it contains
print(poem_text_lines[100])


## List Slicing  (Works with strings too!)

Before we move to the next step of our poetry generation process, let's take a look at one piece of Python from the code above because it is super handy.

Recall that we _index_ into a list using square brackets `[]` with a number in between.  For example the last line of the previous cell prints the line at index 100 in the list `poem_text_lines`.

The previous cell also contains the following line of code:

`raw_lines = lines[34:2654]`

Instead of a single number in between the square brackets we have two numbers separated by a colon. This allows us to take a _slice_ from the list.  In this case `raw_lines` contains a list that has the elements from `lines` starting at index 34 and going up to but not including index 2654. How did we know which lines to choose? We opened the file and checked by hand.

Let's try some simple examples:

In [ ]:
word_list = ["why", "lov'st", "thou", "that", "which", "thou", "receiv'st", "not", "gladly"]

# For each of the following lines, try writing a comment once you see what it 
# does, or try to predict the behaviour.

print(word_list[0:2]) 

print(word_list[:2])
print(word_list[6:8])
print(word_list[:-1])
print(word_list[3:-2])

In [ ]:
# Slicing works the same way on strings

word = word_list[-3]

# Here are some examples
print(word[0:3])
print(word[1:-1])
print(word[-2:])


## 2. Looking up a pronunciation

We did this last week, and once again, you don't need to worry about understanding the code below.

In [ ]:
# We also need to be able to look up a pronunciation given a word.  This
# is the same code that we used last week. You still don't need to worry
# about fully understanding it.

# The first three lines import the nltk library and set up to use the cmudictionary
import nltk
nltk.download('cmudict')
from nltk.corpus import cmudict
# Create a Python dictionary from the cmudict and store it in pronounce_dictionary
pronounce_dictionary = cmudict.dict()

def get_pronunciation(word):
    ''' Return the first phoneme list found for the word word
    '''

    # Ensure the word we want to look up is in lower case
    word = word.lower()

    # We check first to make sure that word is in the dictionary to 
    # avoid getting a nasty error message
    if word in pronounce_dictionary:

        # If the word is in the dictionary then return its first pronunciation list
        return pronounce_dictionary[word][0]
    else:
        # If it is not in the dictionary return the value None
        return None

## 3. Getting a stress pattern

First let's review the code we ended with last week. There are two versions below: one with a comment for each line to review what each line does, and one with minimal comments that might be easier to read once you understand the syntax.

In [ ]:
# Store the string "elephant" in the variable word
word = "elephant"

# Store the list reprepsenting the phonetic pronunciations of elepahnt in word_p
word_p = ['EH1', 'L', 'AH0', 'F', 'AH0', 'N', 'T']

# The variable stress_pattern will be used to build up the stress pattern of 
# the word. We assign it an empty string to start, but will add characters to it
stress_pattern = "" # empty string to start

# Take an action for each element in the list word_p
# The variable phoneme holds the next element of the list in each iteration of
# the loop. In the first iteration of the loop the value of phoneme will be 
# 'EH1'
for phoneme in word_p:

    # Set last_character to hold the last character in phoneme.
    # The square brackets mean we are indexing into the string stored in phoneme
    # -1 means we start counting from the end of the string
    last_character = phoneme[-1]

    # .isdigit() is a string function that returns true if all characters in the
    # string are digits.  (last_character is a string of one character)
    if last_character.isdigit():

        # If it is a digit then we want to add that digit to the end of stress_patthern
        stress_pattern = stress_pattern + last_character

# Because the indentation is the same as the "for" line, we are no longer in 
# the loop os the following print statement will only happen once.
# This is an "f" string where the expressions in the curly braces, {}, are 
# replaced by their values
print(f"The stress pattern of {word} is {stress_pattern}")

In [ ]:
# This is the same code as the previous cell, but without all of the comments

word = "elephant"
word_p = ['EH1', 'L', 'AH0', 'F', 'AH0', 'N', 'T']

# Extract the stress pattern from the pronunciation.

stress_pattern = "" # empty string to start

for phoneme in word_p:
    last_character = phoneme[-1]
    if last_character.isdigit():
        stress_pattern = stress_pattern + last_character
print(f"The stress pattern of {word} is {stress_pattern}")

## Defining a function

We are going to want to get the stress pattern for many words and we don't want to keep writing these same lines of code over and over. Instead we will define a function that we can call. 

You already have experience using functions such at `len()`, `type()`, and even `get_pronunciation()`. `get_pronunciation()` is a _user-defined_ function, and you can see its definition above. Now we will take a look at what goes into such a definition.

When we call a function, it takes one or more arguments (which we put in between the the parentheses), and returns a value.  In the case of the function `len()`, it takes one argument, and the function returns an integer representing the length of that argument.

So let's begin to write a function called `get_stress_pattern()`:
- What is the argument (or arumgents) to the function?
- What does the function return?

Here is a comment describing the function. This is a different kind of notation for a comment, and the function description comment comes after the first line defining the function.

<pre>
''' Return the stress pattern as a string for the word given as an argument.

    Example: get_stress_pattern("awake") returns "01"
'''
</pre>


The definition of a function starts with the keyword `def` and we have to name all of the arguments between the parentheses.  The `def` line ends with a colon and all the code executed by the function (the _body_ of the function) is indented just like the body of a loop.

Notice the `return` statement at the end of the function.  This is how the function knows which value to give back or _return_.  A function may have multiple return statements, but once a return statement is executed, the function ends.

There are two additional lines of code in this function that didn't appear above:

<pre>
    if not word_p:
        return None
</pre>

It is possible, even likely, that we will ask for the pronunciation of words that don't exist in the CMU dictionary, so we need to return a special value in this case. `None` is the Python way of saying we don't have a valid answer. It's important to handle this case otherwise our code will crash. You can try commenting out these two lines of code below to see what happens.

In [ ]:
def get_stress_pattern(word):
    ''' Return the stress pattern as a string for the word given as an argument.

        Example: get_stress_pattern("awake") returns "01"
    '''
    # the body of the function is indented
    stress_pattern = "" # empty string to start
    word_p = get_pronunciation(word)

    # If there isn't a pronunciation then return None
    if not word_p:
        return None

    for phoneme in word_p:
        last_character = phoneme[-1]
        if last_character.isdigit():
            stress_pattern = stress_pattern + last_character
    return stress_pattern

## Approach 1: Use single word iambs 

We are finally ready to start writing code to generate a poem.
We already know how to identify words that have the stress pattern of an iamb: "01", so let's create a list of single-word iambs.  Then each line of the sonnet will be 5 randomly chosen iambs.

### Aside: The "in" operator

Above we ran some code to generate a list of poem lines, which are now stored in a variable called `poem_text_lines`. For this approach we want a list of words rather than lines, so let's first look at how we can create a list of unique words given the list of lines from the poems.

We need to talk about the `in` operator. It isn't actually new because we have already seen it in `for` loops, but it is pretty handy.

The expression `a in b` evaluates to `True` if `a` is an element of `b`.  


Here we will use it to determine whether a word is already in a list of words. This is most commonly used in conditional statements.

In [ ]:
is_in_word = 'o' in 'hello'

print(is_in_word)

In [ ]:
body_parts = ['head', 'shoulder', 'knee', 'toe']
is_item = 'knuckle' in body_parts

print(is_item)

In [ ]:
part = 'knuckle'

if part in body_parts:
    print(f"{part} is in the list of body parts" )
else:
    print(f"{part} is not in the list of body parts")

In [ ]:
# If part is not already in body parts, then add it
if part not in body_parts:
    body_parts.append(part)

Now let's apply the same technique to our list of poem lines.

In [ ]:
# Create an empty list to store unique words from poem_text_lines
word_bank = []
for line in poem_text_lines:
    # split the line into a list of words (splitting on whitespace)
    line_words = line.split()

    for word in line_words:
        if word not in word_bank:
            word_bank.append(word)

# How many words are in our word bank?

print(len(word_bank)) 


The next step is to create a list that contains only single word iambs. That is words that are two syllables where the second syllable is stressed.

There is an additional wrinkle here. Some of the words in our word bank might not be in the pronunciation dictionary, and in this case `get_stress_pattern()` will return the special value `None`.  Since `None` is does not match `"01"` or `"11"` words not in the pronunciation dictionary will not get into our list of iambs.

In [ ]:
iambs = []

for word in word_bank:

    pattern = get_stress_pattern(word)

    if pattern == "01":
        iambs.append(word)
    if pattern == "11":
        iambs.append(word)

print(len(iambs))

# What does the next line do?
iambs[:20]

Now lets create a "sonnet" without rhymes.

There are two new things here:
- We are importing the modeule `random`, but this time we are using a method called `random.sample(list, k)` that selects`k` elements randomly from `list`.  In the example below, 5 words are selected at random from the iambs list.

- We need to make string out of a list of words and to do this we use a string method: `str.join(list)`.  This method looks a bit odd, so you need to read it carefully to understand. Think of the initial string `str` as the string that will be put in between all the words.

Most of the time we will use `str.join(list)` to insert spaces in between the elements of the list.

In [ ]:
import random 

# Select k random elements from the list. Try rerunning this cell with 
# different number values or different lists.
some_words = random.sample(iambs, 5)

print(some_words)

In [ ]:
# An example to illustrate the .join() method

word_list = ["Here", "is", "a", "silly", "example."]

# Create a string by joining the words in word_list with " "
print(" ".join(word_list))

# Create a string by joining the words in word_list with " like "
joiner = " like "
print(joiner.join(word_list))



Finally we are ready to generate 14 lines of iambic pentameter.  (I hesitate to call this a poem.)

# Approach #2 Bi-grams

We aren't going to get very far with poems made up of five words on each line, so the question is how can we select words differently so lines start to look more like English grammar.

One possibility would be to actually analyze the grammar of the source text and use grammatical labels to build up lines, but that's too complicated for today. However, you might want to consider something like this for your project.

Instead we will look for two word sequences (bi-grams) in our source text that are iambs. 

As an aside, n-grams which are sequences of n things (words, characters, symbols) are a basic building block in Natural Language Processing algorithms. N-grams preserve the sequence of words and retain some context, which makes them useful in a wide range of problems including text prediction and translation.

We have to do a little more work to identify 2-word sequences in our source text. At this point we have two useful lists that contain our source material of Shakespeare's sonnets:
- `poem_text_lines` - A list of poem lines with all the blank lines and lines containing the poem numbers removed.
- `word_bank` - A list of the unique words in the poems

The list `poem_text_lines` preserves the order of the words, so we can use this list to build our list of bigrams.

Let's look at a specific example to help us figure out how to design the code

![Bigrams diagram](bigrams.jpg)


In [ ]:
# Given the following line, make a list of words using .split()
line = "mine be thy love and thy love's use their treasure"

line_words = line.split()

print(line_words)

In [ ]:
# Let's build the first three bigrams by hand and then we will try to generalize

bigram_one_list = [line_words[0], line_words[1]]
bigram_two_list = [line_words[1], line_words[2]]
bigram_three_list = [line_words[2], line_words[3]]

bigram_one = " ".join(bigram_one_list)
bigram_two = " ".join(bigram_two_list)
bigram_three = " ".join(bigram_three_list)

print(bigram_one)
print(bigram_two)
print(bigram_three)

The next step is to build a list of all of the bigrams in one line. You can see the pattern we created by hand in the diagram and in the code above.

We are going to use a for loop, but instead of iterating over each element of the list we want to iterate over the indices of the list. In other words we want our loop variable to be the index into the list rather than an element of the list.

To do this we need a list of the values of the indices, and there is a function called `range` to help us:

In [ ]:
for i in range(0, 10):
    print(i)

In [ ]:
for index in range(0, len(line_words)):
    print(line_words[index])


In [ ]:
# Build up a list of all of the bigrams in a line

line_bigram_list = []
for index in range(0, len(line_words) - 1):
    bigram = [line_words[index], line_words[index + 1]]
    line_bigram_list.append(bigram)

print(line_bigram_list)


In [ ]:
# Put it all together to build a bigram list of all of Shakespeare's Sonnets

bigram_list = []

for line in poem_text_lines:
    line_words = line.split()
    for index in range(0, len(line_words) - 1):
        bigram = [line_words[index], line_words[index + 1]]
        bigram_list.append(bigram)

print(f"{len(bigram_list)} bigrams were found")

The next step is to build a list of bigrams that are iambs. This time we will also turn them into strings, so that we can ultimately join them together.

In [ ]:
iamb_bigrams = []
for bigram in bigram_list:
    word1 = bigram[0]
    word2 = bigram[1]

    stress_word1 = get_stress_pattern(word1)
    stress_word2 = get_stress_pattern(word2)

    # Make sure we have a valid stress pattern for both words
    if stress_word1 and stress_word2:
        if stress_word1 == "0" and stress_word2 == "1":
            iamb_bigrams.append(" ".join(bigram))
        if stress_word1 == "1" and stress_word2 == "1":
            iamb_bigrams.append(" ".join(bigram))



print(len(iamb_bigrams), "of them are natural iambs")
print(random.sample(iamb_bigrams, 10))

In [ ]:
# combine the two lists of iambs, but we want them all to be strings.
all_iambs = iamb_bigrams + iambs
print(len(all_iambs))

In [ ]:
for i in range(0, 14):
    line_chunks = random.sample(all_iambs, 5)
    print(" ".join(line_chunks))

## Extra: Can we make it rhyme?

There is no new Python in the code below, but we would have to work through some additional logic.

The key observation is that two words rhyme when the phonemes of the last syllabe are the same.  This is a slice of the list from last vowel to the end of the phoneme list.
This code still needs some refinement to get better results.

In [ ]:
def rhyme_part(word):
    phonemes = get_pronunciation(word)
    i = len(phonemes) - 1
    while i > 0:
        # if the last digit of the phoneme is a digit
        # then we have found the last vowel sound
        if phonemes[i][-1].isdigit():
            return phonemes[i:]   # from the last stressed/vowel sound to the end
        i -= 1
    return phonemes

def rhymes_with(target_word, bank):
    matches = []
    for word in bank:
        if rhyme_part(word) == rhyme_part(target_word):
            if word != target_word:
                matches.append(word)
    return matches

line1_words = random.sample(iambs, 5)
line1 = " ".join(line1_words)

candidates = rhymes_with(line1_words[-1], iambs)

if len(candidates) == 0:
    print("No rhymes for ", line1_words[-1])
else:
    line2_words = random.sample(iambs, 4) + [random.choice(candidates)]
    line2 = " ".join(line2_words)

    print(line1)
    print(line2)